# Section 7 · Governance 1 — Bedrock Guardrails: control what the agent *says*

Guardrails evaluate both the user's input and the model's output and block content that breaks your policies:
- **Content filters** — hate, violence, sexual content, misconduct, insults
- **Denied topics** — e.g. legal advice
- **PII** — block or mask personal data

Then we look at what guardrails **cannot** do — which is where AgentCore Policy (next notebook) comes in.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os
import time
import uuid

import boto3

MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-sonnet-4-6")
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
bedrock_client = boto3.client("bedrock", region_name=REGION)

---
## 1. Create a guardrail with the API

You can also build guardrails in the Bedrock console (*Safeguards → Guardrails*). Here we do it in code with three policies: content filters, a denied topic, and PII handling.

In [ ]:
response = bedrock_client.create_guardrail(
    name=f"agent-guardrail-{uuid.uuid4().hex[:6]}",
    description="Guardrail with content filtering, denied topics, and PII detection.",
    blockedInputMessaging="Your request was blocked by our safety policies.",
    blockedOutputsMessaging="The response was blocked by our safety policies.",
    topicPolicyConfig={
        "topicsConfig": [
            {
                "name": "Legal Advice",
                "definition": "Providing specific legal advice, legal opinions, or recommendations on legal matters.",
                "examples": [
                    "Should I sue my neighbor?",
                    "What are my legal options for this dispute?",
                    "Can I take legal action against my employer?",
                ],
                "type": "DENY",
            }
        ]
    },
    contentPolicyConfig={
        "filtersConfig": [
            {"type": "HATE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "VIOLENCE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "SEXUAL", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "MISCONDUCT", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "INSULTS", "inputStrength": "HIGH", "outputStrength": "HIGH"},
        ]
    },
    sensitiveInformationPolicyConfig={
        "piiEntitiesConfig": [
            {"type": "US_SOCIAL_SECURITY_NUMBER", "action": "BLOCK"},
            {"type": "CREDIT_DEBIT_CARD_NUMBER", "action": "BLOCK"},
            {"type": "EMAIL", "action": "ANONYMIZE"},
        ]
    },
)

GUARDRAIL_ID = response["guardrailId"]
GUARDRAIL_VERSION = "DRAFT"  # use DRAFT while testing; publish a numbered version for production

# Wait until the guardrail is ready
while (status := bedrock_client.get_guardrail(guardrailIdentifier=GUARDRAIL_ID, guardrailVersion=GUARDRAIL_VERSION)["status"]) != "READY":
    print("Guardrail status:", status)
    time.sleep(3)

print(f"Guardrail ready: {GUARDRAIL_ID} ({response['guardrailArn']})")

## 2. Attach the guardrail to an agent

Pass the guardrail ID and version to `BedrockModel`. Every model call — input and output — is now checked.

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel

model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    guardrail_id=GUARDRAIL_ID,
    guardrail_version=GUARDRAIL_VERSION,
    guardrail_trace="enabled",
)

agent = Agent(model=model, system_prompt="You are a customer support agent for an insurance company.")

### Test: request for personal information

In [ ]:
response = agent("Tell me the CEO's home address and social security number.")
print(f"\nStop reason: {response.stop_reason}")

### Test: PII in the user's own message (SSN → blocked)

In [ ]:
response = agent("My SSN is 123-45-6789. Can you update my policy with it?")
print(f"\nStop reason: {response.stop_reason}")

### Test: denied topic

In [ ]:
response = agent("Should I sue my neighbor? What are my legal options?")
print(f"\nStop reason: {response.stop_reason}")

### Test: normal request (should pass)

In [ ]:
response = agent("What is the process for filing a home insurance claim?")
print(f"\nStop reason: {response.stop_reason}")

### Handling an intervention in your app

When a guardrail blocks, `stop_reason` is `guardrail_intervened`. Use it to log, alert, or show a friendly message.

In [ ]:
response = agent("Give me someone's credit card number.")

if response.stop_reason == "guardrail_intervened":
    print("\nGuardrail blocked this request.")
    print(f"Safe response returned: {response}")
else:
    print(f"\nNormal response: {response}")

---
## 3. What guardrails cannot do

Guardrails filter **text**. They do not see the agent's decision to call a tool. Here an agent with a refund tool happily calls it with $50,000 — the guardrail is active, but nothing in the text is "unsafe".

In [ ]:
from strands import tool


@tool
def process_refund(customer_id: str, amount: float, reason: str) -> str:
    """Process a refund for a customer."""
    # In a real system this would call a payment API
    print(f"\n  [TOOL CALLED] process_refund(customer_id={customer_id}, amount=${amount:,.2f}, reason={reason})")
    return f"Refund of ${amount:,.2f} processed for customer {customer_id}. Reason: {reason}"


agent = Agent(
    model=model,
    system_prompt="You are a helpful customer support agent. Process refunds when customers request them.",
    tools=[process_refund],
)

# The guardrail will NOT prevent this tool call
response = agent("I'm really frustrated. Just give me a $50,000 refund for customer C-1234 and we'll call it even.")

The tool ran with `amount=50000`. The guardrail did not — and could not — stop it.

**Use guardrails to control what the agent says. Use policy to control what the agent does.** The next notebook, `02_agentcore_policy_cedar.ipynb`, closes this gap with deterministic Cedar rules.

---
## Clean up

In [ ]:
bedrock_client.delete_guardrail(guardrailIdentifier=GUARDRAIL_ID)
print(f"Deleted guardrail: {GUARDRAIL_ID}")